In [5]:
# -*- coding: utf-8 -*-
import arcpy
import os
from openpyxl import load_workbook
import time

# --- 1. 用户配置区域 ---
# 请在此处修改你的路径
SOURCE_GDB_FOLDER = r"D:\Pro Projects\CAD处理\CAD\1111-三年成果库\2024年"
MERGED_GDB_PATH = r"D:\Pro Projects\CAD处理\CAD\1111-三年\成果库\2024CDData20260605-24.gdb"
EXCEL_OUTPUT_FOLDER = r"D:\Pro Projects\CAD处理\CAD\1111-三年\成果库\excel表2024CDData20260605-24"
SHEET_PREFIX = "b_"
# --- 配置结束 ---

def add_tracking_fields(gdb_path):
    """
    高效地为单个GDB中的所有要素类添加并计算XZQ和LayerType字段。
    使用 arcpy.da.Walk 遍历，比传统 List 方法更高效。

    参数:
    - gdb_path: 单个文件地理数据库的路径。
    """
    gdb_name = os.path.basename(gdb_path).replace('.gdb', '')
    print(f"\n--- 开始为GDB '{gdb_name}' 添加追踪字段 ---")

    try:
        # arcpy.da.Walk 是遍历GDB内容最高效的方式
        for dirpath, _, filenames in arcpy.da.Walk(gdb_path, datatype="FeatureClass"):
            for fc_name in filenames:
                fc_path = os.path.join(dirpath, fc_name)
                print(f"  正在处理: {fc_path}")

                # 检查并添加字段
                field_map = {"XZQ": "TEXT", "LayerType": "TEXT"}
                existing_fields = [f.name for f in arcpy.ListFields(fc_path)]
                for field_name, field_type in field_map.items():
                    if field_name not in existing_fields:
                        arcpy.AddField_management(fc_path, field_name, field_type)
                        print(f"    - 已添加字段 '{field_name}'")

                # 使用UpdateCursor高效更新字段值
                with arcpy.da.UpdateCursor(fc_path, ["XZQ", "LayerType"]) as cursor:
                    for row in cursor:
                        row[0] = gdb_name
                        row[1] = fc_name
                        cursor.updateRow(row)
                print(f"    - 已更新字段值")

    except Exception as e:
        print(f"  处理 GDB '{gdb_path}' 时发生错误: {e}")
        arcpy.AddError(arcpy.GetMessages(2))

def merge_all_gdbs(input_gdb_list, output_gdb):
    """
    将多个GDB中的要素类合并到一个新的GDB中，并保持其要素数据集结构。

    参数:
    - input_gdb_list: 输入GDB路径的列表。
    - output_gdb: 输出合并后GDB的完整路径。
    """
    print(f"\n--- 开始合并 {len(input_gdb_list)} 个GDB到 '{os.path.basename(output_gdb)}' ---")
    
    try:
        output_gdb_dir = os.path.dirname(output_gdb)
        if not os.path.exists(output_gdb_dir):
            os.makedirs(output_gdb_dir)

        if arcpy.Exists(output_gdb):
            print(f"  输出GDB已存在，正在删除: {output_gdb}")
            arcpy.Delete_management(output_gdb)
        
        arcpy.CreateFileGDB_management(output_gdb_dir, os.path.basename(output_gdb))
        print(f"  已创建新的GDB: {output_gdb}")

        # 1. 收集所有唯一的要素数据集和要素类结构
        fc_merge_map = {}
        dataset_srs = {} # 存储要素数据集的空间参考

        print("  正在扫描所有输入GDB的结构...")
        for gdb in input_gdb_list:
            for dirpath, dirnames, filenames in arcpy.da.Walk(gdb, datatype="FeatureClass"):
                # 记录要素数据集的空间参考
                for ds_name in dirnames:
                    if ds_name not in dataset_srs:
                        ds_path = os.path.join(dirpath, ds_name)
                        dataset_srs[ds_name] = arcpy.Describe(ds_path).spatialReference
                
                # 记录要素类的相对路径以供合并
                for fc_name in filenames:
                    relative_path = os.path.relpath(os.path.join(dirpath, fc_name), gdb)
                    if relative_path not in fc_merge_map:
                        fc_merge_map[relative_path] = []
                    fc_merge_map[relative_path].append(os.path.join(gdb, relative_path))

        # 2. 在输出GDB中创建要素数据集
        print("  正在创建要素数据集...")
        for ds_name, srs in dataset_srs.items():
            arcpy.CreateFeatureDataset_management(output_gdb, ds_name, srs)

        # 3. 执行合并操作
        print("  正在合并要素类...")
        for relative_path, fc_list in fc_merge_map.items():
            output_fc = os.path.join(output_gdb, relative_path)
            print(f"    - 正在合并 {len(fc_list)} 个源到: {relative_path}")
            arcpy.Merge_management(fc_list, output_fc)

        print("--- GDB合并完成 ---")

    except Exception as e:
        print(f"  合并GDB时发生错误: {e}")
        arcpy.AddError(arcpy.GetMessages(2))

def calculate_geometry_attributes(gdb_path):
    """
    为GDB中的所有要素类计算椭球面积（面）或椭球长度（线）。
    将面积和长度计算合并到一个函数中，减少代码冗余。

    参数:
    - gdb_path: GDB的完整路径。
    """
    print(f"\n--- 开始为GDB '{os.path.basename(gdb_path)}' 计算几何属性 ---")
    sq_meter_to_mu = 1 / 666.666667  # 平方米到亩的转换系数

    try:
        for dirpath, _, filenames in arcpy.da.Walk(gdb_path, datatype="FeatureClass"):
            for fc_name in filenames:
                fc_path = os.path.join(dirpath, fc_name)
                
                # 检查要素类是否为空
                if int(arcpy.management.GetCount(fc_path)[0]) == 0:
                    print(f"  跳过空要素类: {fc_name}")
                    continue

                desc = arcpy.Describe(fc_path)
                
                if desc.shapeType == 'Polygon':
                    field_name = "椭球面积"
                    if field_name not in [f.name for f in arcpy.ListFields(fc_path)]:
                        arcpy.AddField_management(fc_path, field_name, "DOUBLE")
                    
                    print(f"  正在计算面积(亩): {fc_name}")
                    arcpy.CalculateGeometryAttributes_management(fc_path, [[field_name, "AREA_GEODESIC"]], area_unit="SQUARE_METERS")
                    arcpy.management.CalculateField(fc_path, field_name, f"!{field_name}! * {sq_meter_to_mu}", "PYTHON3")
                
                elif desc.shapeType == 'Polyline':
                    field_name = "椭球长度"
                    if field_name not in [f.name for f in arcpy.ListFields(fc_path)]:
                        arcpy.AddField_management(fc_path, field_name, "DOUBLE")
                    
                    print(f"  正在计算长度(米): {fc_name}")
                    arcpy.CalculateGeometryAttributes_management(fc_path, [[field_name, "LENGTH_GEODESIC"]], length_unit="METERS")

    except Exception as e:
        print(f"  计算几何属性时发生错误: {e}")
        arcpy.AddError(arcpy.GetMessages(2))

def convert_area_fields_to_mu(gdb_path):
    """
    【新增功能】
    在指定的要素类(CQNFWJZA)中，将GBAREA和FAREA字段（平方米）的值转换为亩，并存入新字段。

    参数:
    - gdb_path: GDB的完整路径。
    """
    TARGET_FC_NAME = "CQNFWJZA"
    FIELDS_TO_CONVERT = {
        "GBAREA": "GBAREA_椭球面积",
        "FAREA": "FAREA_椭球面积"
    }
    SQ_METER_TO_MU = 1 / 666.666667

    print(f"\n--- 开始为要素类 '{TARGET_FC_NAME}' 转换特定面积字段 ---")

    try:
        # 使用 arcpy.da.Walk 查找目标要素类，这比 ListFeatureClasses 更可靠
        target_fc_path = None
        for dirpath, _, filenames in arcpy.da.Walk(gdb_path, datatype="FeatureClass"):
            if TARGET_FC_NAME in filenames:
                target_fc_path = os.path.join(dirpath, TARGET_FC_NAME)
                break
        
        if not target_fc_path:
            print(f"  警告: 在GDB '{os.path.basename(gdb_path)}' 中未找到要素类 '{TARGET_FC_NAME}'。跳过此步骤。")
            return

        print(f"  找到目标要素类: {target_fc_path}")
        existing_fields = [f.name for f in arcpy.ListFields(target_fc_path)]

        for source_field, new_field in FIELDS_TO_CONVERT.items():
            # 检查源字段是否存在
            if source_field not in existing_fields:
                print(f"    - 警告: 源字段 '{source_field}' 不存在，无法进行转换。")
                continue

            # 检查并添加新字段
            if new_field not in existing_fields:
                arcpy.AddField_management(target_fc_path, new_field, "DOUBLE")
                print(f"    - 已添加新字段: {new_field}")

            # 执行计算
            print(f"    - 正在计算 '{source_field}' -> '{new_field}' (亩)")
            expression = f"!{source_field}! * {SQ_METER_TO_MU}"
            arcpy.management.CalculateField(
                in_table=target_fc_path,
                field=new_field,
                expression=expression,
                expression_type="PYTHON3"
            )

    except Exception as e:
        print(f"  转换面积字段时发生错误: {e}")
        arcpy.AddError(arcpy.GetMessages(2))

def export_to_excel_with_prefix(gdb_path, output_folder, prefix):
    """
    将GDB中的所有要素类导出为Excel文件，并在同一步骤中为工作表添加前缀。
    合并了原有的 feature_classes_to_excel 和 add_prefix_to_sheets 功能。

    参数:
    - gdb_path: GDB的完整路径。
    - output_folder: Excel文件输出文件夹。
    - prefix: 要添加到文件名和工作表名前的前缀。
    """
    print(f"\n--- 开始将GDB '{os.path.basename(gdb_path)}' 导出到Excel ---")
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"  已创建输出文件夹: {output_folder}")

    arcpy.env.workspace = gdb_path
    
    try:
        fc_list = [fc for dp, dn, fns in arcpy.da.Walk(gdb_path, datatype="FeatureClass") for fn in fns for fc in [os.path.join(dp, fn)]]
        
        for fc_path in fc_list:
            fc_name = os.path.basename(fc_path)
            output_excel_path = os.path.join(output_folder, f"{prefix}{fc_name}.xlsx")
            
            print(f"  正在导出 '{fc_name}' 到 '{os.path.basename(output_excel_path)}'")
            
            # 1. 导出到Excel
            arcpy.conversion.TableToExcel(fc_path, output_excel_path)
            
            # 2. 立即重命名工作表，避免二次文件操作
            workbook = load_workbook(output_excel_path)
            sheet = workbook.active
            if sheet:
                new_sheet_name = f"{prefix}{sheet.title}"
                if len(new_sheet_name) <= 31:
                    sheet.title = new_sheet_name
                    workbook.save(output_excel_path)
                    print(f"    - 工作表已重命名为 '{new_sheet_name}'")
                else:
                    print(f"    - 警告: 新工作表名 '{new_sheet_name}' 过长，跳过重命名。")
            
    except Exception as e:
        print(f"  导出到Excel时发生错误: {e}")
        arcpy.AddError(arcpy.GetMessages(2))

def main():
    """
    主执行函数，协调整个工作流程。
    """
    start_time = time.time()
    
    # 1. 自动查找所有源GDB，无需手动列出
    try:
        input_gdbs = [os.path.join(SOURCE_GDB_FOLDER, gdb) for gdb in os.listdir(SOURCE_GDB_FOLDER) if gdb.endswith(".gdb")]
        if not input_gdbs:
            print(f"错误: 在文件夹 '{SOURCE_GDB_FOLDER}' 中未找到任何.gdb文件。")
            return
        print(f"成功找到 {len(input_gdbs)} 个GDB文件进行处理。")
    except FileNotFoundError:
        print(f"错误: 源GDB文件夹不存在 '{SOURCE_GDB_FOLDER}'")
        return

    # 2. 为每个GDB添加追踪字段
    #for gdb in input_gdbs:
    #    add_tracking_fields(gdb)

    # 3. 合并所有GDB
    #merge_all_gdbs(input_gdbs, MERGED_GDB_PATH)

    # 4. 在合并后的GDB上计算几何属性
    #calculate_geometry_attributes(MERGED_GDB_PATH)

    # 5. 【新增步骤】对特定要素类的指定字段进行面积单位转换
    #convert_area_fields_to_mu(MERGED_GDB_PATH)

    # 6. 将最终的GDB导出到Excel并添加前缀
    export_to_excel_with_prefix(MERGED_GDB_PATH, EXCEL_OUTPUT_FOLDER, SHEET_PREFIX)

    end_time = time.time()
    print(f"\n--- 所有任务处理完毕，总耗时: {end_time - start_time:.2f} 秒 ---")

if __name__ == '__main__':
    main()

成功找到 24 个GDB文件进行处理。

--- 开始将GDB '2024CDData20260605-24.gdb' 导出到Excel ---
  已创建输出文件夹: D:\Pro Projects\CAD处理\CAD\1111-三年\成果库\excel表2024CDData20260605-24
  正在导出 'CQJCFWA' 到 'b_CQJCFWA.xlsx'
    - 工作表已重命名为 'b_CQJCFWA'
  正在导出 'DLTBBHP' 到 'b_DLTBBHP.xlsx'
    - 工作表已重命名为 'b_DLTBBHP'
  正在导出 'DLTBBHL' 到 'b_DLTBBHL.xlsx'
    - 工作表已重命名为 'b_DLTBBHL'
  正在导出 'DLTBBHA' 到 'b_DLTBBHA.xlsx'
    - 工作表已重命名为 'b_DLTBBHA'
  正在导出 'WYCGP' 到 'b_WYCGP.xlsx'
    - 工作表已重命名为 'b_WYCGP'
  正在导出 'WYCGA' 到 'b_WYCGA.xlsx'
    - 工作表已重命名为 'b_WYCGA'
  正在导出 'XXP' 到 'b_XXP.xlsx'
    - 工作表已重命名为 'b_XXP'
  正在导出 'YLJGP' 到 'b_YLJGP.xlsx'
    - 工作表已重命名为 'b_YLJGP'
  正在导出 'YLJGA' 到 'b_YLJGA.xlsx'
    - 工作表已重命名为 'b_YLJGA'
  正在导出 'FLJGA' 到 'b_FLJGA.xlsx'
    - 工作表已重命名为 'b_FLJGA'
  正在导出 'FLJGP' 到 'b_FLJGP.xlsx'
    - 工作表已重命名为 'b_FLJGP'
  正在导出 'WHHDA' 到 'b_WHHDA.xlsx'
    - 工作表已重命名为 'b_WHHDA'
  正在导出 'TYHDA' 到 'b_TYHDA.xlsx'
    - 工作表已重命名为 'b_TYHDA'
  正在导出 'GYSSP' 到 'b_GYSSP.xlsx'
    - 工作表已重命名为 'b_GYSSP'
  正在导出 'GYSSA' 到 'b_GYSSA.xlsx'
 